In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
抓取内蒙古招生考试信息网“专业组”链接对应的专业录取明细，并导出 Excel。

默认目标页：
https://www.nm.zsks.cn/25gktdlq/zdxlqzgdf-bk-wl-qcsj-1/tj/lqzgzdf.html#
"""

from __future__ import annotations

import argparse
import json
import re
import ssl
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any
from urllib.parse import urlencode, urljoin, urldefrag
from urllib.request import Request, urlopen

try:
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.utils import get_column_letter
except ImportError as exc:
    raise SystemExit("缺少依赖 openpyxl，请先运行：python3 -m pip install openpyxl") from exc


DEFAULT_URL = (
    "https://www.nm.zsks.cn/25gktdlq/"
    "zdxlqzgdf-bk-wl-qcsj-1/tj/lqzgzdf.html#"
)

JSON_URL_RE = re.compile(r"url\s*:\s*['\"](?P<url>[^'\"]+\.json)['\"]", re.I)


GROUP_COLUMNS = [
    ("PCDM", "批次代码"),
    ("PCMC", "批次"),
    ("JHLBMC", "计划类别"),
    ("KLDM", "科类代码"),
    ("KLMC", "科类"),
    ("YXDH", "院校代号"),
    ("YXMC", "院校名称"),
    ("ZYZDH", "专业组"),
    ("ZYZLQRS", "专业组录取人数"),
    ("ZYZZGF", "专业组最高分"),
    ("ZYZZDF", "专业组最低分"),
    ("LINK", "专业组链接"),
]

DETAIL_COLUMNS = [
    ("PCDM", "批次代码"),
    ("PCMC", "批次"),
    ("JHLBMC", "计划类别"),
    ("KLDM", "科类代码"),
    ("KLMC", "科类"),
    ("YXDH", "院校代号"),
    ("YXMC", "院校名称"),
    ("ZYZDH", "专业组"),
    ("ZYZLQRS", "专业组录取人数"),
    ("ZYZZGF", "专业组最高分"),
    ("ZYZZDF", "专业组最低分"),
    ("ZYDH", "专业代号"),
    ("ZYMC", "专业名称"),
    ("LQRS", "专业录取人数"),
    ("ZGF", "专业最高分"),
    ("ZDF", "专业最低分"),
    ("LINK", "专业组链接"),
]

TEXT_FIELDS = {"PCDM", "KLDM", "YXDH", "ZYZDH", "ZYDH", "LINK"}
NUMBER_FIELDS = {"ZYZLQRS", "ZYZZGF", "ZYZZDF", "LQRS", "ZGF", "ZDF"}


def fetch_bytes(url: str, timeout: int) -> tuple[bytes, str | None]:
    request = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/125.0 Safari/537.36"
            ),
            "Accept": "text/html,application/json;q=0.9,*/*;q=0.8",
        },
    )
    context = None
    if url.lower().startswith("https://"):
        context = ssl.create_default_context()
        # 该站点的 TLS 参数偏旧；降低 OpenSSL 安全等级以兼容 Python 3.13+。
        try:
            context.set_ciphers("DEFAULT:@SECLEVEL=1")
        except ssl.SSLError:
            pass

    with urlopen(request, timeout=timeout, context=context) as response: